# Numerical agreement with SciPy

Speed is only useful if the distributions remain numerically trustworthy.
This notebook compares Baldr's scalar, NumPy, and JAX backends with
`scipy.stats` for `logpdf`, `pdf`, `cdf`, and `ppf`.

Central values, support boundaries, and tail probabilities are evaluated
separately. PPF accuracy is also checked by the round trip
`cdf(ppf(q))`, which tests the inverse in the way prior transforms use it.

In [ ]:
import importlib.metadata
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from baldr import (
    Beta,
    Exponential,
    Gamma,
    Normal,
    TruncatedNormal,
    Uniform,
)

try:
    import jax
    import jax.numpy as jnp
except ImportError:
    jax = None

np.set_printoptions(precision=4, suppress=False)

In [ ]:
def package_version(name):
    """Return an installed package version.

    Parameters
    ----------
    name : str
        Distribution package name.

    Returns
    -------
    str
        Installed version or "not installed".
    """
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"


pd.Series(
    {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        **{
            name: package_version(name)
            for name in ("baldr", "numpy", "scipy", "jax", "jaxlib")
        },
    },
    name="version",
)

In [ ]:
SPECS = {
    "Normal": {
        "baldr": lambda backend: Normal(
            loc=0.3, scale=1.7, backend=backend
        ),
        "scipy": stats.norm(loc=0.3, scale=1.7),
        "x": np.array([-8.2, -3.1, 0.3, 3.7, 8.8]),
    },
    "Uniform": {
        "baldr": lambda backend: Uniform(
            loc=-1.2, scale=3.4, backend=backend
        ),
        "scipy": stats.uniform(loc=-1.2, scale=3.4),
        "x": np.array([-1.3, -1.2, 0.5, 2.2, 2.3]),
    },
    "Beta": {
        "baldr": lambda backend: Beta(
            a=2.3, b=5.1, loc=-0.2, scale=1.4, backend=backend
        ),
        "scipy": stats.beta(
            a=2.3, b=5.1, loc=-0.2, scale=1.4
        ),
        "x": np.array([-0.21, -0.2, 0.0, 0.6, 1.2, 1.21]),
    },
    "Exponential": {
        "baldr": lambda backend: Exponential(
            scale=1.6, backend=backend
        ),
        "scipy": stats.expon(scale=1.6),
        "x": np.array([-0.1, 0.0, 0.2, 2.0, 20.0]),
    },
    "Gamma": {
        "baldr": lambda backend: Gamma(
            a=2.7, loc=0.1, scale=1.3, backend=backend
        ),
        "scipy": stats.gamma(a=2.7, loc=0.1, scale=1.3),
        "x": np.array([0.0, 0.1, 0.2, 2.0, 20.0]),
    },
    "TruncatedNormal": {
        "baldr": lambda backend: TruncatedNormal(
            loc=0.2,
            scale=1.1,
            low=-1.4,
            high=2.7,
            backend=backend,
        ),
        "scipy": stats.truncnorm(
            (-1.4 - 0.2) / 1.1,
            (2.7 - 0.2) / 1.1,
            loc=0.2,
            scale=1.1,
        ),
        "x": np.array([-1.5, -1.4, 0.2, 2.7, 2.8]),
    },
}

METHODS = ("logpdf", "pdf", "cdf")
PROBABILITIES = np.array(
    [
        1e-12,
        1e-9,
        1e-6,
        1e-3,
        0.1,
        0.5,
        0.9,
        1 - 1e-3,
        1 - 1e-6,
        1 - 1e-9,
        1 - 1e-12,
    ]
)

## Error metrics

Absolute error is meaningful near zero; relative error is meaningful
away from zero. For logarithmic densities we primarily use absolute
error. Non-finite values match only when both implementations return the
same signed infinity or both return `NaN`.

In [ ]:
def finite_errors(actual, reference):
    """Summarize finite and special-value agreement.

    Parameters
    ----------
    actual : array-like
        Values produced by Baldr.
    reference : array-like
        Reference values produced by SciPy.

    Returns
    -------
    tuple
        Maximum absolute error, maximum relative error, and whether non-finite
        values match.
    """
    actual = np.asarray(actual, dtype=float)
    reference = np.asarray(reference, dtype=float)
    finite = np.isfinite(actual) & np.isfinite(reference)
    absolute = (
        np.max(np.abs(actual[finite] - reference[finite]))
        if np.any(finite)
        else 0.0
    )
    denominator = np.maximum(np.abs(reference[finite]), 1e-300)
    relative = (
        np.max(
            np.abs(actual[finite] - reference[finite]) / denominator
        )
        if np.any(finite)
        else 0.0
    )
    special_match = np.all(
        (actual[~finite] == reference[~finite])
        | (np.isnan(actual[~finite]) & np.isnan(reference[~finite]))
    )
    return absolute, relative, special_match


def as_numpy(value):
    """Convert an evaluated result to a floating-point NumPy array.

    Parameters
    ----------
    value : array-like
        Result to convert.

    Returns
    -------
    numpy.ndarray
        Converted floating-point array.
    """
    return np.asarray(value, dtype=float)

## NumPy backend against `scipy.stats`

The input grids deliberately include points immediately outside and exactly on
finite support boundaries.

For the Uniform example, `loc=-1.2` and `scale=3.4` give an upper endpoint
of `2.2`. SciPy may standardize that decimal endpoint to a floating-point
value slightly greater than one and return zero density or `-inf` log-density,
while Baldr compares directly with `loc + scale` and includes the endpoint.
The resulting Uniform endpoint row is therefore reported as a boundary-rounding
artefact rather than a general numerical disagreement.

In [ ]:
rows = []
for distribution_name, spec in SPECS.items():
    baldr_distribution = spec["baldr"]("numpy")
    scipy_distribution = spec["scipy"]

    for method_name in METHODS:
        actual = getattr(baldr_distribution, method_name)(spec["x"])
        reference = getattr(scipy_distribution, method_name)(spec["x"])
        absolute, relative, special_match = finite_errors(
            actual, reference
        )
        rows.append(
            {
                "distribution": distribution_name,
                "backend": "numpy",
                "method": method_name,
                "region": "support_grid",
                "max_absolute_error": absolute,
                "max_relative_error": relative,
                "special_values_match": special_match,
            }
        )

    actual = baldr_distribution.ppf(PROBABILITIES)
    reference = scipy_distribution.ppf(PROBABILITIES)
    absolute, relative, special_match = finite_errors(actual, reference)
    rows.append(
        {
            "distribution": distribution_name,
            "backend": "numpy",
            "method": "ppf",
            "region": "tails",
            "max_absolute_error": absolute,
            "max_relative_error": relative,
            "special_values_match": special_match,
        }
    )

    round_trip = baldr_distribution.cdf(actual)
    absolute, relative, special_match = finite_errors(
        round_trip, PROBABILITIES
    )
    rows.append(
        {
            "distribution": distribution_name,
            "backend": "numpy",
            "method": "cdf(ppf(q))",
            "region": "tails",
            "max_absolute_error": absolute,
            "max_relative_error": relative,
            "special_values_match": special_match,
        }
    )

agreement = pd.DataFrame(rows)
agreement

In [ ]:
agreement.pivot_table(
    index=["distribution", "method"],
    columns="backend",
    values="max_absolute_error",
).style.format("{:.3e}")

## Scalar backend against SciPy

Scalar checks call one value at a time, matching the backend's intended
use. The same grids are retained so boundary behaviour is compared
directly with the vector implementation.

In [ ]:
scalar_rows = []
for distribution_name, spec in SPECS.items():
    baldr_distribution = spec["baldr"]("scalar")
    scipy_distribution = spec["scipy"]

    for method_name in METHODS:
        actual = [
            getattr(baldr_distribution, method_name)(float(value))
            for value in spec["x"]
        ]
        reference = getattr(scipy_distribution, method_name)(spec["x"])
        absolute, relative, special_match = finite_errors(
            actual, reference
        )
        scalar_rows.append(
            {
                "distribution": distribution_name,
                "backend": "scalar",
                "method": method_name,
                "region": "support_grid",
                "max_absolute_error": absolute,
                "max_relative_error": relative,
                "special_values_match": special_match,
            }
        )

    actual = [
        baldr_distribution.ppf(float(q)) for q in PROBABILITIES
    ]
    reference = scipy_distribution.ppf(PROBABILITIES)
    absolute, relative, special_match = finite_errors(actual, reference)
    scalar_rows.append(
        {
            "distribution": distribution_name,
            "backend": "scalar",
            "method": "ppf",
            "region": "tails",
            "max_absolute_error": absolute,
            "max_relative_error": relative,
            "special_values_match": special_match,
        }
    )

    round_trip = [
        baldr_distribution.cdf(value) for value in actual
    ]
    absolute, relative, special_match = finite_errors(
        round_trip, PROBABILITIES
    )
    scalar_rows.append(
        {
            "distribution": distribution_name,
            "backend": "scalar",
            "method": "cdf(ppf(q))",
            "region": "tails",
            "max_absolute_error": absolute,
            "max_relative_error": relative,
            "special_values_match": special_match,
        }
    )

agreement = pd.concat(
    [agreement, pd.DataFrame(scalar_rows)], ignore_index=True
)
agreement.query("backend == 'scalar'")

## JAX float32 and float64 agreement

JAX precision is process-wide. Each dtype is configured explicitly, and
results are converted to NumPy only after evaluation. Float32 should not
be judged against float64 tolerances, especially in PPF tails.

In [ ]:
jax_rows = []
if jax is not None:
    original_x64 = jax.config.x64_enabled
    try:
        for dtype_name, dtype in (
            ("float32", jnp.float32),
            ("float64", jnp.float64),
        ):
            jax.config.update("jax_enable_x64", dtype_name == "float64")
            for distribution_name, spec in SPECS.items():
                distribution = spec["baldr"]("jax")
                scipy_distribution = spec["scipy"]
                x = jnp.asarray(spec["x"], dtype=dtype)

                for method_name in METHODS:
                    function = jax.jit(
                        getattr(distribution, method_name)
                    )
                    actual = as_numpy(function(x))
                    reference = getattr(
                        scipy_distribution, method_name
                    )(spec["x"])
                    absolute, relative, special_match = finite_errors(
                        actual, reference
                    )
                    jax_rows.append(
                        {
                            "distribution": distribution_name,
                            "backend": f"jax_{dtype_name}",
                            "method": method_name,
                            "region": "support_grid",
                            "max_absolute_error": absolute,
                            "max_relative_error": relative,
                            "special_values_match": special_match,
                        }
                    )

                q = jnp.asarray(PROBABILITIES, dtype=dtype)
                ppf = jax.jit(distribution.ppf)
                actual = as_numpy(ppf(q))
                reference = scipy_distribution.ppf(PROBABILITIES)
                absolute, relative, special_match = finite_errors(
                    actual, reference
                )
                jax_rows.append(
                    {
                        "distribution": distribution_name,
                        "backend": f"jax_{dtype_name}",
                        "method": "ppf",
                        "region": "tails",
                        "max_absolute_error": absolute,
                        "max_relative_error": relative,
                        "special_values_match": special_match,
                    }
                )

                round_trip = as_numpy(
                    jax.jit(distribution.cdf)(
                        jnp.asarray(actual, dtype=dtype)
                    )
                )
                absolute, relative, special_match = finite_errors(
                    round_trip, PROBABILITIES
                )
                jax_rows.append(
                    {
                        "distribution": distribution_name,
                        "backend": f"jax_{dtype_name}",
                        "method": "cdf(ppf(q))",
                        "region": "tails",
                        "max_absolute_error": absolute,
                        "max_relative_error": relative,
                        "special_values_match": special_match,
                    }
                )
    finally:
        jax.config.update("jax_enable_x64", original_x64)

    agreement = pd.concat(
        [agreement, pd.DataFrame(jax_rows)], ignore_index=True
    )
else:
    print("JAX is not installed; skipping this section.")

agreement

In [ ]:
ppf_errors = agreement.query("method == 'ppf'")
figure, axis = plt.subplots(figsize=(10, 5))
for backend, group in ppf_errors.groupby("backend"):
    axis.semilogy(
        group["distribution"],
        np.maximum(group["max_absolute_error"], 1e-18),
        marker="o",
        label=backend,
    )
axis.set_ylabel("Maximum absolute PPF error")
axis.set_title("Tail-quantile agreement with scipy.stats")
axis.tick_params(axis="x", rotation=30)
axis.legend()
figure.tight_layout()

## Inspecting where a tail error occurs

A single maximum can hide whether an error is isolated to an extreme
tail. This helper plots Baldr-minus-SciPy PPF residuals against
probability for any selected distribution and backend.

In [ ]:
def plot_ppf_residuals(distribution_name="Beta", backend="numpy"):
    """Plot inverse-CDF residuals relative to SciPy.

    Parameters
    ----------
    distribution_name : str, optional
        Distribution key from SPECS.
    backend : {"scalar", "numpy", "jax"}, optional
        Baldr backend to evaluate.
    """
    spec = SPECS[distribution_name]
    distribution = spec["baldr"](backend)
    q = PROBABILITIES
    if backend == "jax":
        actual = as_numpy(
            jax.jit(distribution.ppf)(jnp.asarray(q))
        )
    elif backend == "scalar":
        actual = np.array(
            [distribution.ppf(float(value)) for value in q]
        )
    else:
        actual = as_numpy(distribution.ppf(q))
    reference = spec["scipy"].ppf(q)

    figure, axis = plt.subplots(figsize=(8, 4))
    axis.semilogx(q, actual - reference, marker="o")
    axis.set_xlabel("Probability")
    axis.set_ylabel("Baldr - SciPy")
    axis.set_title(f"{distribution_name} PPF residuals ({backend})")
    figure.tight_layout()


plot_ppf_residuals("Beta", "numpy")

## Interpretation

- Inspect absolute and relative errors together.
- Boundary infinities are correct only when their sign matches SciPy.
- For prior transforms, `cdf(ppf(q))` is often the clearest operational
  accuracy measure.
- Expect float32 tail accuracy to degrade before float64.
- A fast implementation that misses the required tail tolerance should
  not be used merely because central-value agreement is good.